# 🏗️ Building Change Detection from Aerial Imagery
## Siamese SegFormer-B2 + Attention-Based Fusion on WHU-CD Dataset

---

### 🎯 Project Overview
This notebook implements an **end-to-end Siamese Change Detection pipeline** for detecting building changes between two aerial images taken at different times. Given two 512×512 aerial photos of the same location (before and after), the model predicts a binary change mask — white pixels indicate changed buildings, black pixels indicate no change.

This project directly extends the **Building Footprint Extraction** project. The pretrained SegFormer-B2 encoder is reused as a shared Siamese encoder — initializing from a domain-specific checkpoint rather than ImageNet alone.

**Dataset:** WHU Building Change Detection Dataset — Christchurch, New Zealand
- **Image A (2012):** 12,796 buildings — post-earthquake damage
- **Image B (2016):** 16,077 buildings — reconstruction underway
- **Change Mask:** White = changed (added / demolished / modified buildings)
- **Class balance:** ~6% changed pixels across all splits

---

### 🧠 Model — Siamese SegFormer-B2 with Change Attention Module

```
Image A (2012) → Shared MiT-B2 Encoder → Features A ─┐
                                                        → ChangeAttentionModule → MLP Decoder → Change Map
Image B (2016) → Shared MiT-B2 Encoder → Features B ─┘
                    ↑ same weights, initialized from pretrained footprint model
```

**Key novelties:**
- **Domain-specific pretraining** — encoder initialized from footprint model, not ImageNet
- **ChangeAttentionModule** — learns *which channels* and *which spatial locations* matter most for change, replacing naive |F_A − F_B|

---

### 📉 Loss Function

```
L = 0.35 · FocalLoss + 0.40 · DiceLoss + 0.25 · BoundaryLoss
```

- **Focal Loss** (α=0.75, γ=2.0) — handles severe class imbalance (only ~6% changed pixels)
- **Dice Loss** — optimizes overlap directly
- **Boundary Loss** — Laplacian edge-weighting for sharp change boundaries

---

### 📊 Test Results

| Metric | Score |
|--------|-------|
| **IoU** | **0.6196** ± 0.1958 |
| **Dice / F1** | **0.7458** ± 0.1619 |
| **Precision** | 0.7322 |
| **Recall** | 0.7696 |

---

### 🗂️ Notebook Structure
1. 📦 Imports
2. ⬇️ Install Dependencies
3. 🔗 Mount Drive & Copy Pretrained Weights
4. ⚙️ Configuration
5. ⬇️ Download & Preprocess Dataset
6. 📂 Create Validation Split
7. 🖼️ Visualize Dataset
8. 🗃️ Dataset Class & DataLoaders
9. 🧠 Model — Siamese SegFormer-B2
10. 📉 Loss Function
11. 📊 Metrics
12. 🏋️ Train & Eval Epochs
13. 👁️ Visualize Predictions
14. 🚀 Main Training Pipeline
15. 🏆 Test Evaluation

## 📦 Imports

Key libraries:
- `SegformerModel` — raw encoder only (not the segmentation head), so we can build our own Siamese decoder on top
- `albumentations` — augmentation with `additional_targets` support, critical for applying identical transforms to both images
- `rasterio` — reading GeoTIFF files from the WHU dataset

In [ ]:
import os
import numpy as np
from PIL import Image
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import albumentations as A
from albumentations.pytorch import ToTensorV2
from transformers import SegformerModel
import torch.nn.functional as F
from tqdm.auto import tqdm
import matplotlib.pyplot as plt

## ⬇️ Install Dependencies

In [ ]:
!pip install -q transformers albumentations rasterio

## 🔗 Mount Drive & Copy Pretrained Weights

Mounts Google Drive and copies the pretrained footprint encoder (`best_model.pth`) to the Colab working directory.

This file is the output of the **Building Footprint Extraction** project — its encoder weights are transferred to the Siamese model here.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
import shutil

# Copy from Drive to Colab working directory
src = '/content/drive/MyDrive/best_model.pth'
dst = '/content/best_model.pth'

shutil.copy(src, dst)

# Verify
size = os.path.getsize(dst) / 1024 / 1024
print(f"✅ Copied successfully!")
print(f"📦 File size: {size:.2f} MB")

# Should be ~90-100MB for mit-b2
if size < 10:
    print("⚠️ File seems too small — might be corrupted or wrong file")
else:
    print("✅ Size looks correct — ready to train!")

## ⚙️ Configuration

| Parameter | Value | Reason |
|-----------|-------|--------|
| `batch_size` | 8 | Safe for Colab T4 GPU |
| `lr` | 1e-4 | Peak LR for OneCycleLR — encoder gets 10× lower |
| `epochs` | 50 | With early stopping (patience=15) |
| `model_name` | mit-b2 | Same encoder as footprint model |
| `pretrained_seg` | best_model.pth | Domain-specific encoder initialization |
| `patience` | 15 | Early stopping threshold |

**Differential LR:** encoder trains at `lr × 0.1`, all decoder components at `lr`. This prevents destroying pretrained features early in training.

In [ ]:
def get_device():
    if torch.cuda.is_available():
        return "cuda"
    elif torch.backends.mps.is_available():
        return "mps"
    return "cpu"

CONFIG = {
    "data_root": "/content/WHU_CD_processed",
    "batch_size":     8,
    "num_workers":    2,
    "lr":             1e-4,
    "epochs":         50,
    "device":         get_device(),
    "save_path": "/content/drive/MyDrive/best_cd_model.pth",
    "model_name":     "nvidia/mit-b2",
    # Path to your pretrained footprint encoder — set None to train from scratch
    "pretrained_seg": "/content/best_model.pth",
    "patience":       15,
    "img_size":       512,
}

print(f"Using device: {CONFIG['device']}")
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None'}")

## ⬇️ Download & Preprocess Dataset

Downloads the WHU Building Change Detection dataset from the official WHU GPCV server.

**What this cell does:**
1. Downloads and extracts the raw dataset (~5.4 GB)
2. Reads 2012 and 2016 label masks per tile using `rasterio`
3. Computes a binary change mask: `change = (label_2012 != label_2016)`
4. Copies 2012 images → `A/`, 2016 images → `B/`, change masks → `label/`

**Change mask logic:**
- Pixel = building in 2012 but not 2016 → **demolished** → changed
- Pixel = not building in 2012 but building in 2016 → **new construction** → changed
- Pixel unchanged in both → no change

In [ ]:
import os

# مسیر ذخیره
zip_path = "/content/WHU_CD.zip"
extract_path = "/content/WHU_CD"

os.makedirs(extract_path, exist_ok=True)

# دانلود دیتاست
!wget -O "{zip_path}" "https://gpcv.whu.edu.cn/data/Building%20change%20detection%20dataset_add.zip"

# بررسی حجم فایل
!ls -lh "{zip_path}"

# استخراج
!unzip -q "{zip_path}" -d "{extract_path}"

print("Dataset ready at:", extract_path)

# نمایش ساختار
!find "{extract_path}" -maxdepth 2 -type d

In [ ]:
import os
import shutil
import rasterio
import numpy as np
from tqdm import tqdm

ROOT = "/content/WHU_CD/Building change detection dataset_add/1. The two-period image data"

OUT = "/content/WHU_CD_processed"

# ساخت فولدرها
for split in ["train", "test"]:
    for folder in ["A", "B", "label"]:
        os.makedirs(
            os.path.join(OUT, split, folder),
            exist_ok=True
        )


def process_split(split):

    img_2012_dir = os.path.join(
        ROOT, "2012", "splited_images", split, "image"
    )

    label_2012_dir = os.path.join(
        ROOT, "2012", "splited_images", split, "label"
    )

    img_2016_dir = os.path.join(
        ROOT, "2016", "splited_images", split, "image"
    )

    label_2016_dir = os.path.join(
        ROOT, "2016", "splited_images", split, "label"
    )

    out_A = os.path.join(OUT, split, "A")
    out_B = os.path.join(OUT, split, "B")
    out_label = os.path.join(OUT, split, "label")

    # فایل‌های مشترک
    files = sorted(
        set(os.listdir(img_2012_dir))
        & set(os.listdir(label_2012_dir))
        & set(os.listdir(img_2016_dir))
        & set(os.listdir(label_2016_dir))
    )

    # فقط tif
    files = [
        f for f in files
        if f.lower().endswith((".tif", ".tiff"))
    ]

    print(f"\n{split}: {len(files)} samples")

    for fname in tqdm(files):

        path_2012 = os.path.join(img_2012_dir, fname)
        path_2016 = os.path.join(img_2016_dir, fname)

        path_label_2012 = os.path.join(label_2012_dir, fname)
        path_label_2016 = os.path.join(label_2016_dir, fname)

        # -------------------------
        # Copy RGB images
        # -------------------------

        shutil.copy2(
            path_2012,
            os.path.join(out_A, fname)
        )

        shutil.copy2(
            path_2016,
            os.path.join(out_B, fname)
        )

        # -------------------------
        # Generate change mask
        # -------------------------

        with rasterio.open(path_label_2012) as src:
            label_2012 = src.read(1)

        with rasterio.open(path_label_2016) as src:
            label_2016 = src.read(1)

        # 0/255 -> boolean change
        change = (label_2012 != label_2016)

        # تبدیل به uint8 با مقادیر 0 و 255
        change = (change.astype(np.uint8) * 255)

        # ذخیره Change Mask
        with rasterio.open(
            os.path.join(out_label, fname),
            "w",
            driver="GTiff",
            height=change.shape[0],
            width=change.shape[1],
            count=1,
            dtype="uint8"
        ) as dst:

            dst.write(change, 1)


# پردازش train و test
process_split("train")
process_split("test")

print("\n✅ Dataset preprocessing completed!")
print("Output:", OUT)

## 📂 Create Validation Split

The original dataset only provides `train` and `test` splits. This cell carves out **10% of training data** as a validation set using a fixed random seed (42) for reproducibility.

**Final split sizes:**
- Train: 1,134 tiles
- Val: 126 tiles
- Test: 690 tiles

**Class balance (% changed pixels):**
- Train: 6.46%
- Val: 6.18%
- Test: 5.11%

Balanced enough that no resampling is needed — the Focal Loss handles this imbalance.

In [ ]:
import os
import shutil
import random
from tqdm import tqdm

ROOT = "/content/WHU_CD_processed"

random.seed(42)

# مسیرها
train_A = os.path.join(ROOT, "train", "A")
train_B = os.path.join(ROOT, "train", "B")
train_label = os.path.join(ROOT, "train", "label")

val_A = os.path.join(ROOT, "val", "A")
val_B = os.path.join(ROOT, "val", "B")
val_label = os.path.join(ROOT, "val", "label")

# ساخت validation directories
for path in [val_A, val_B, val_label]:
    os.makedirs(path, exist_ok=True)

# فایل‌های مشترک
files = sorted(
    set(os.listdir(train_A))
    & set(os.listdir(train_B))
    & set(os.listdir(train_label))
)

files = [
    f for f in files
    if f.lower().endswith((".tif", ".tiff"))
]

print("Total train samples:", len(files))

# 10% برای validation
random.shuffle(files)

val_size = int(0.1 * len(files))

val_files = files[:val_size]

print("Validation samples:", len(val_files))
print("Remaining training samples:", len(files) - len(val_files))

# انتقال فایل‌های validation
for fname in tqdm(val_files, desc="Creating validation set"):

    shutil.move(
        os.path.join(train_A, fname),
        os.path.join(val_A, fname)
    )

    shutil.move(
        os.path.join(train_B, fname),
        os.path.join(val_B, fname)
    )

    shutil.move(
        os.path.join(train_label, fname),
        os.path.join(val_label, fname)
    )

print("\n✅ Validation set created!")

In [ ]:
for split in ["train", "val", "test"]:
    print(
        f"{split}:",
        "A =", len(os.listdir(f"{ROOT}/{split}/A")),
        "| B =", len(os.listdir(f"{ROOT}/{split}/B")),
        "| label =", len(os.listdir(f"{ROOT}/{split}/label"))
    )

## 🖼️ Visualize Dataset

Displays random sample triplets: **Image A (Before) · Image B (After) · Change Mask**

Used to verify:
1. Both images are correctly paired (same geographic tile)
2. Change mask is binary (white = changed, black = unchanged)
3. Changed regions visually correspond to new or demolished buildings

In [ ]:
import random

def show_cd_samples(split='train', n=3):
    dir_A     = os.path.join(CONFIG["data_root"], split, 'A')
    dir_B     = os.path.join(CONFIG["data_root"], split, 'B')
    dir_label = os.path.join(CONFIG["data_root"], split, 'label')

    files = sorted(set(os.listdir(dir_A)) & set(os.listdir(dir_B)) & set(os.listdir(dir_label)))
    if not files:
        print(f"No files found in {split} split — check your folder structure")
        return

    selected = random.sample(files, min(n, len(files)))
    fig, axes = plt.subplots(n, 3, figsize=(15, 5 * n))
    if n == 1:
        axes = axes.reshape(1, -1)

    for i, fname in enumerate(selected):
        img_A = np.array(Image.open(os.path.join(dir_A,     fname)).convert("RGB"))
        img_B = np.array(Image.open(os.path.join(dir_B,     fname)).convert("RGB"))
        label = np.array(Image.open(os.path.join(dir_label, fname)))

        axes[i, 0].imshow(img_A);            axes[i, 0].set_title(f"A — Before ({fname})"); axes[i, 0].axis('off')
        axes[i, 1].imshow(img_B);            axes[i, 1].set_title("B — After");             axes[i, 1].axis('off')
        axes[i, 2].imshow(label, cmap='gray'); axes[i, 2].set_title("Change Mask");          axes[i, 2].axis('off')

        changed_pct = (label > 0).mean() * 100
        print(f"{fname}: changed pixels = {changed_pct:.2f}%")

    plt.tight_layout()
    plt.show()

show_cd_samples('train', n=3)

## 🗃️ Dataset Class & DataLoaders

### `WHUChangeDetectionDataset`
Loads **triplets** of (Image A, Image B, Change Label) with matching filenames across `A/`, `B/`, `label/`.

### Critical augmentation detail
Both images must receive **identical** spatial transforms — a flip applied to Image A must also be applied to Image B, otherwise the comparison becomes meaningless. Albumentations handles this via `additional_targets`:

```python
additional_targets = {"image2": "image"}
transform(image=img_A, image2=img_B, mask=label)
```

### Train augmentations
| Transform | Probability | Purpose |
|-----------|-------------|---------|
| HorizontalFlip | 0.5 | Orientation invariance |
| VerticalFlip | 0.5 | Orientation invariance |
| RandomRotate90 | 0.5 | Rotation invariance |
| RandomBrightnessContrast | 0.3 | Lighting robustness |
| GaussNoise / GaussianBlur | 0.2 | Sensor noise |
| RandomFog / RandomShadow / RandomSunFlare | 0.2 | Atmospheric condition differences between 2012 and 2016 imagery |

In [ ]:
class WHUChangeDetectionDataset(Dataset):
    def __init__(self, split='train', data_root=CONFIG["data_root"], transform=None):
        self.dir_A     = os.path.join(data_root, split, 'A')
        self.dir_B     = os.path.join(data_root, split, 'B')
        self.dir_label = os.path.join(data_root, split, 'label')
        self.transform = transform

        files_A     = set(os.listdir(self.dir_A))
        files_B     = set(os.listdir(self.dir_B))
        files_label = set(os.listdir(self.dir_label))
        self.files  = sorted(files_A & files_B & files_label)
        print(f"[{split}] Found {len(self.files)} samples")

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        fname  = self.files[idx]
        img_A  = np.array(Image.open(os.path.join(self.dir_A,     fname)).convert("RGB"))
        img_B  = np.array(Image.open(os.path.join(self.dir_B,     fname)).convert("RGB"))
        label  = np.array(Image.open(os.path.join(self.dir_label, fname)))
        label = (label > 0).astype(np.float32)

        if self.transform:
            augmented = self.transform(
                image=img_A,
                image2=img_B,
                mask=label
            )

            img_A = augmented["image"]
            img_B = augmented["image2"]
            label = augmented["mask"]

        if isinstance(label, np.ndarray):
            label = torch.from_numpy(label)

        label = label.float()

        if label.max() > 1:
            label = label / 255.0

        if label.ndim == 2:
            label = label.unsqueeze(0)

        return img_A, img_B, label

def get_transforms(split='train'):
    additional_targets = {"image2": "image"}
    if split == 'train':
        return A.Compose([
            A.HorizontalFlip(p=0.5),
            A.VerticalFlip(p=0.5),
            A.RandomRotate90(p=0.5),
            A.RandomBrightnessContrast(p=0.3),
            A.OneOf([
                A.GaussNoise(p=0.5),
                A.GaussianBlur(p=0.5),
            ], p=0.2),

            A.OneOf([
                A.RandomFog(p=0.3),
                A.RandomShadow(p=0.3),
                A.RandomSunFlare(p=0.2),
            ], p=0.2),
            A.Normalize(mean=(0.485, 0.456, 0.406),
                        std=(0.229, 0.224, 0.225)),
            ToTensorV2(),
        ], additional_targets=additional_targets)
    else:
        return A.Compose([
            A.Normalize(mean=(0.485, 0.456, 0.406),
                        std=(0.229, 0.224, 0.225)),
            ToTensorV2(),
        ], additional_targets=additional_targets)


def get_dataloaders():
    train_ds = WHUChangeDetectionDataset('train', transform=get_transforms('train'))
    val_ds   = WHUChangeDetectionDataset('val',   transform=get_transforms('val'))
    test_ds  = WHUChangeDetectionDataset('test',  transform=get_transforms('test'))

    train_loader = DataLoader(train_ds, batch_size=CONFIG["batch_size"],
                              shuffle=True,  num_workers=CONFIG["num_workers"], pin_memory=False)
    val_loader   = DataLoader(val_ds,   batch_size=CONFIG["batch_size"],
                              shuffle=False, num_workers=CONFIG["num_workers"], pin_memory=False)
    test_loader  = DataLoader(test_ds,  batch_size=CONFIG["batch_size"],
                              shuffle=False, num_workers=CONFIG["num_workers"], pin_memory=False)
    return train_loader, val_loader, test_loader

## 🧠 Model — Siamese SegFormer-B2

### Architecture

```
Image A → Shared MiT-B2 Encoder → [F_A1, F_A2, F_A3, F_A4]  (4 scales)
                                                                     ↓
                                       ChangeAttentionModule per scale
                                       (channel attention + spatial attention)
                                                                     ↓
                                    Project all scales → 256-dim + upsample to H/4
                                                                     ↓
                                        Concatenate → Conv+BN+ReLU → Dropout
                                                                     ↓
                                               1×1 Conv → Upsample → (B,1,512,512)
Image B → Shared MiT-B2 Encoder → [F_B1, F_B2, F_B3, F_B4]
```

### ChangeAttentionModule
Replaces naive `|F_A − F_B|` with a learned attention mechanism:

1. **Channel Attention** — learns which feature channels are most informative for change
2. **Spatial Attention** — learns which spatial locations within those features matter most
3. **Fusion** — concatenates F_A, F_B, and attended difference → 1×1 Conv

### Pretrained Encoder Transfer
```python
# Keys in footprint model:  "segformer.stages.X...."
# Keys in SegformerModel:   "stages.X...."
# Fix: strip "segformer." prefix and load_state_dict(strict=False)
```
All 364 encoder weights are transferred successfully from the footprint model.

### SegFormer-B2 Channel Dimensions
| Stage | Channels | Spatial size (512×512 input) |
|-------|----------|------------------------------|
| 1 | 64 | 128×128 |
| 2 | 128 | 64×64 |
| 3 | 320 | 32×32 |
| 4 | 512 | 16×16 |

In [ ]:
class ChangeAttentionModule(nn.Module):
    """
    Attention-based feature fusion for change detection.

    Instead of using only:
        |F_A - F_B|

    the module learns:
        1. Which channels are important
        2. Which spatial locations are important

    Then fuses:
        F_A, F_B, attended_difference
    """

    def __init__(self, channels):
        super().__init__()

        # --------------------------------------------------
        # Channel Attention
        # --------------------------------------------------
        # Input:
        #   [B, 2C]
        #
        # Output:
        #   [B, C]
        # --------------------------------------------------
        self.channel_attention = nn.Sequential(
            nn.Linear(channels * 2, channels // 4),
            nn.ReLU(inplace=True),
            nn.Linear(channels // 4, channels),
            nn.Sigmoid()
        )

        # --------------------------------------------------
        # Spatial Attention
        # --------------------------------------------------
        # Input:
        #   [B, 2, H, W]
        #
        # Output:
        #   [B, 1, H, W]
        # --------------------------------------------------
        self.spatial_attention = nn.Sequential(
            nn.Conv2d(
                2,
                1,
                kernel_size=7,
                padding=3,
                bias=False
            ),
            nn.Sigmoid()
        )

        # --------------------------------------------------
        # Feature Fusion
        # --------------------------------------------------
        # Concatenate:
        #   F_A        -> C
        #   F_B        -> C
        #   diff_sa    -> C
        #
        # Total:
        #   3C
        #
        # Output:
        #   C
        # --------------------------------------------------
        self.fusion = nn.Sequential(
            nn.Conv2d(
                channels * 3,
                channels,
                kernel_size=1,
                bias=False
            ),
            nn.BatchNorm2d(channels),
            nn.ReLU(inplace=True),
        )

    def forward(self, fA, fB):

        # --------------------------------------------------
        # 1. Absolute feature difference
        # --------------------------------------------------
        diff = torch.abs(fA - fB)

        # --------------------------------------------------
        # 2. Channel Attention
        # --------------------------------------------------
        # Global average pooling manually
        # [B, C, H, W] -> [B, C]
        avg_A = fA.mean(dim=(2, 3))
        avg_B = fB.mean(dim=(2, 3))

        # [B, C] + [B, C] -> [B, 2C]
        channel_descriptor = torch.cat(
            [avg_A, avg_B],
            dim=1
        )

        # [B, 2C] -> [B, C]
        ca = self.channel_attention(
            channel_descriptor
        )

        # [B, C] -> [B, C, 1, 1]
        ca = ca.unsqueeze(-1).unsqueeze(-1)

        # Apply channel attention
        diff_ca = diff * ca

        # --------------------------------------------------
        # 3. Spatial Attention
        # --------------------------------------------------
        avg_out = torch.mean(
            diff_ca,
            dim=1,
            keepdim=True
        )

        max_out = torch.max(
            diff_ca,
            dim=1,
            keepdim=True
        )[0]

        # [B, 2, H, W]
        spatial_descriptor = torch.cat(
            [avg_out, max_out],
            dim=1
        )

        # [B, 2, H, W] -> [B, 1, H, W]
        sa = self.spatial_attention(
            spatial_descriptor
        )

        # Apply spatial attention
        diff_sa = diff_ca * sa

        # --------------------------------------------------
        # 4. Feature Fusion
        # --------------------------------------------------
        fused = torch.cat(
            [fA, fB, diff_sa],
            dim=1
        )

        return self.fusion(fused)

class SiameseSegFormerCD(nn.Module):
    def __init__(self, model_name="nvidia/mit-b2"):
        super().__init__()

        self.encoder = SegformerModel.from_pretrained(
            model_name,
            output_hidden_states=True,
        )

        enc_channels = {
            "nvidia/mit-b0": [32,  64,  160, 256],
            "nvidia/mit-b2": [64,  128, 320, 512],
        }
        self.enc_channels = enc_channels.get(model_name, [64, 128, 320, 512])

        # ← جایگزین fusion_convs قبلی
        self.fusion_convs = nn.ModuleList([
            ChangeAttentionModule(c)
            for c in self.enc_channels
        ])

        decoder_dim = 256
        self.decoder_projections = nn.ModuleList([
            nn.Conv2d(c, decoder_dim, kernel_size=1, bias=False)
            for c in self.enc_channels
        ])

        self.decoder_fuse = nn.Sequential(
            nn.Conv2d(decoder_dim * 4, decoder_dim, kernel_size=1, bias=False),
            nn.BatchNorm2d(decoder_dim),
            nn.ReLU(inplace=True),
            nn.Dropout2d(0.1),
        )

        self.classifier = nn.Conv2d(decoder_dim, 1, kernel_size=1)

    def encode(self, x):
        outputs = self.encoder(
            pixel_values=x.contiguous(),
            output_hidden_states=True
        )
        return outputs.hidden_states

    def forward(self, img_A, img_B):
        B, C, H, W = img_A.shape

        feats_A = self.encode(img_A)
        feats_B = self.encode(img_B)

        # ← تغییر: فقط fA و fB می‌فرستیم، diff داخل module محاسبه می‌شه
        fused = []
        for i, (fA, fB) in enumerate(zip(feats_A, feats_B)):
            fused.append(self.fusion_convs[i](fA, fB))

        target_size = (H // 4, W // 4)
        projected = []
        for i, f in enumerate(fused):
            p = self.decoder_projections[i](f).float()
            if img_A.device.type == "mps":
                p = p.cpu()
                p = F.interpolate(p, size=target_size,
                                  mode='bilinear', align_corners=False)
                p = p.to(img_A.device)
            else:
                p = F.interpolate(p, size=target_size,
                                  mode='bilinear', align_corners=False)
            projected.append(p)

        x = torch.cat(projected, dim=1)
        x = self.decoder_fuse(x)
        x = self.classifier(x)

        if img_A.device.type == "mps":
            x = x.cpu()
            x = F.interpolate(x, size=(H, W),
                              mode='bilinear', align_corners=False)
            x = x.to(img_A.device)
        else:
            x = F.interpolate(x, size=(H, W),
                              mode='bilinear', align_corners=False)

        return x

### Model Initialization with Pretrained Weights

In [ ]:
def get_model(model_name=CONFIG["model_name"], device=None):
    if device is None:
        device = torch.device(CONFIG["device"])

    model = SiameseSegFormerCD(model_name=model_name).to(device)

    if CONFIG["pretrained_seg"] and os.path.exists(CONFIG["pretrained_seg"]):
        print(f"Loading pretrained encoder from: {CONFIG['pretrained_seg']}")
        pretrained = torch.load(CONFIG["pretrained_seg"],
                                map_location=device, weights_only=True)

        # footprint keys: "segformer.stages.X...."
        # SegformerModel keys: "stages.X...."
        encoder_state = {}
        for k, v in pretrained.items():
            if k.startswith("segformer."):
                encoder_state[k[len("segformer."):]] = v

        missing, unexpected = model.encoder.load_state_dict(encoder_state, strict=False)
        loaded = len(encoder_state) - len(missing)
        print(f"  ✓ Loaded {loaded}/{len(encoder_state)} encoder weights")
        if loaded == len(encoder_state):
            print("  ✅ All encoder weights transferred successfully!")
    else:
        print("No pretrained weights found — training encoder from scratch")

    return model

## 📉 Loss Function — Focal + Dice + Boundary

```
L = 0.35 · FocalLoss + 0.40 · DiceLoss + 0.25 · BoundaryLoss
```

### Why this combination?
| Term | Role |
|------|------|
| **Focal Loss** (α=0.75, γ=2.0) | Handles 94% background / 6% changed class imbalance — down-weights easy negatives |
| **Dice Loss** | Directly optimizes overlap between prediction and ground truth |
| **Boundary Loss** | Laplacian filter extracts change boundaries from GT mask, dilates them, applies 3× weight penalty near edges — forces precise delineation |

### Why no BCE?
Focal Loss is a generalization of BCE that already incorporates class weighting. Using both would double-count the imbalance correction.

In [ ]:
# ── Loss Function ──────────────────────────────────────────

class BoundaryLoss(nn.Module):
    def __init__(self, dilation_radius=3):
        super().__init__()
        self.dilation_radius = dilation_radius
        laplacian = torch.tensor([
            [0,  1, 0],
            [1, -4, 1],
            [0,  1, 0]
        ], dtype=torch.float32).view(1, 1, 3, 3)
        self.register_buffer("laplacian", laplacian)
        d = 2 * dilation_radius + 1
        dilation_kernel = torch.ones(1, 1, d, d, dtype=torch.float32)
        self.register_buffer("dilation_kernel", dilation_kernel)

    def get_boundary_weight_map(self, mask):
        edges = F.conv2d(mask, self.laplacian, padding=1)
        edges = (edges.abs() > 0.1).float()
        boundary_region = F.conv2d(edges, self.dilation_kernel,
                                   padding=self.dilation_radius)
        boundary_region = (boundary_region > 0).float()
        return 1.0 + 2.0 * boundary_region

    def forward(self, logits, targets):
        with torch.no_grad():
            weight_map = self.get_boundary_weight_map(targets)
        bce = F.binary_cross_entropy_with_logits(logits, targets, reduction='none')
        return (bce * weight_map).mean()


class CombinedLoss(nn.Module):
    def __init__(self, focal_w=0.35, dice_w=0.40, boundary_w=0.25,
                 alpha=0.75, gamma=2.0, dilation_radius=3):
        super().__init__()
        self.focal_w    = focal_w
        self.dice_w     = dice_w
        self.boundary_w = boundary_w
        self.alpha      = alpha
        self.gamma      = gamma
        self.boundary   = BoundaryLoss(dilation_radius)

    def focal_loss(self, logits, targets):
        bce   = F.binary_cross_entropy_with_logits(logits, targets, reduction='none')
        probs = torch.sigmoid(logits)
        pt    = torch.where(targets == 1, probs, 1 - probs)
        alpha_t = torch.where(
            targets == 1,
            torch.full_like(targets, self.alpha),
            torch.full_like(targets, 1 - self.alpha)
        )
        return (alpha_t * (1 - pt) ** self.gamma * bce).mean()

    def dice_loss(self, logits, targets):
        probs  = torch.sigmoid(logits)
        smooth = 1e-6
        inter  = (probs * targets).sum(dim=(2, 3))
        dice   = 1 - (2 * inter + smooth) / (
            probs.sum(dim=(2, 3)) + targets.sum(dim=(2, 3)) + smooth)
        return dice.mean()

    def forward(self, logits, targets):
        fl = self.focal_loss(logits, targets)
        dl = self.dice_loss(logits, targets)
        bl = self.boundary(logits, targets)
        total = self.focal_w * fl + self.dice_w * dl + self.boundary_w * bl
        return total, {"focal": fl.item(), "dice": dl.item(), "boundary": bl.item()}

## 📊 Metrics

Four metrics computed per batch, averaged over each epoch.

| Metric | Formula | Note |
|--------|---------|------|
| **IoU** | \|P∩G\| / \|P∪G\| | Primary metric — strict, penalizes both FP and FN |
| **Dice/F1** | 2\|P∩G\| / (\|P\|+\|G\|) | More lenient than IoU |
| **Precision** | TP / (TP+FP) | Low → too many false alarms |
| **Recall** | TP / (TP+FN) | Low → too many missed changes |

For change detection, **recall matters more** — missing a demolished or new building is usually worse than a false alarm.

In [ ]:
def compute_metrics(preds, targets, threshold=0.5):
    preds_bin    = (preds > threshold).float()
    intersection = (preds_bin * targets).sum()
    union        = preds_bin.sum() + targets.sum() - intersection
    iou          = (intersection + 1e-6) / (union + 1e-6)
    dice         = (2 * intersection + 1e-6) / (preds_bin.sum() + targets.sum() + 1e-6)
    tp = (preds_bin * targets).sum()
    fp = (preds_bin * (1 - targets)).sum()
    fn = ((1 - preds_bin) * targets).sum()
    precision = (tp + 1e-6) / (tp + fp + 1e-6)
    recall    = (tp + 1e-6) / (tp + fn + 1e-6)
    return {"iou": iou.item(), "dice": dice.item(),
            "precision": precision.item(), "recall": recall.item()}

## 🏋️ Train Epoch

One complete pass through the training set.

**Per-batch steps:**
1. Move `img_A`, `img_B`, `masks` to device
2. Forward pass through Siamese model: `model(img_A, img_B)`
3. Compute Focal + Dice + Boundary loss
4. Backprop + AdamW step
5. OneCycleLR scheduler step (**per batch**, not per epoch)

**Differential LR strategy:**
- Encoder: `lr × 0.1` — fine-tunes pretrained weights slowly
- Decoder (fusion, projections, classifier): `lr` — trains fast from random init

**Encoder warmup:**
- Epochs 1–5: encoder frozen, only decoder trains
- Epoch 6+: encoder unfrozen, full model trains jointly

In [ ]:
def train_epoch(model, loader, optimizer, criterion, scheduler, device):
    model.train()

    total_loss = 0.0
    metrics_sum = {
        "iou": 0.0,
        "dice": 0.0,
        "precision": 0.0,
        "recall": 0.0
    }

    pbar = tqdm(
        loader,
        desc="Training",
        leave=True,
        dynamic_ncols=False,
        position=0
    )

    for img_A, img_B, masks in pbar:
        img_A = img_A.to(device)
        img_B = img_B.to(device)
        masks = masks.to(device)

        optimizer.zero_grad(set_to_none=True)

        logits = model(img_A, img_B)

        loss, _ = criterion(logits, masks)

        loss.backward()
        optimizer.step()
        scheduler.step()

        total_loss += loss.item()

        with torch.no_grad():
            probs = torch.sigmoid(logits)
            m = compute_metrics(probs, masks)

            for k in metrics_sum:
                metrics_sum[k] += m[k]

        pbar.set_postfix(
            loss=f"{loss.item():.4f}",
            iou=f"{m['iou']:.4f}"
        )

    n = len(loader)

    return (
        total_loss / n,
        {k: v / n for k, v in metrics_sum.items()}
    )

## 🧪 Eval Epoch

No-gradient evaluation loop for validation and test sets.

Uses `@torch.no_grad()` and `model.eval()` — disables dropout and switches BatchNorm to running statistics.

In [ ]:
@torch.no_grad()
def eval_epoch(model, loader, criterion, device):
    model.eval()

    total_loss = 0.0
    metrics_sum = {
        "iou": 0.0,
        "dice": 0.0,
        "precision": 0.0,
        "recall": 0.0
    }

    pbar = tqdm(
        loader,
        desc="Validation",
        leave=True,
        dynamic_ncols=False,
        position=0
    )

    for img_A, img_B, masks in pbar:
        img_A = img_A.to(device)
        img_B = img_B.to(device)
        masks = masks.to(device)

        logits = model(img_A, img_B)

        loss, _ = criterion(logits, masks)

        total_loss += loss.item()

        probs = torch.sigmoid(logits)

        m = compute_metrics(probs, masks)

        for k in metrics_sum:
            metrics_sum[k] += m[k]

        pbar.set_postfix(
            loss=f"{loss.item():.4f}",
            iou=f"{m['iou']:.4f}"
        )

    n = len(loader)

    return (
        total_loss / n,
        {k: v / n for k, v in metrics_sum.items()}
    )

## 👁️ Visualize Predictions

5-column visualization per sample:

| Column | Content |
|--------|---------|
| Before (A) | 2012 aerial image |
| After (B) | 2016 aerial image |
| Ground Truth | Annotated change mask |
| Prediction | Model output at threshold 0.5 |
| Error Map | 🟢 TP · 🔴 FP · 🔵 FN |

The error map shows **what kind** of mistake the model makes — false alarms (red) vs missed changes (blue).

In [ ]:
def visualize_predictions(model, loader, device, n=3):
    model.eval()
    img_A, img_B, masks = next(iter(loader))
    img_A = img_A.to(device)
    img_B = img_B.to(device)

    with torch.no_grad():
        logits = model(img_A, img_B)
        preds  = (torch.sigmoid(logits) > 0.5).float()

    mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
    std  = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)

    fig, axes = plt.subplots(n, 5, figsize=(20, 4 * n))
    titles = ["Before (A)", "After (B)", "Ground Truth", "Prediction", "Error Map"]

    for i in range(n):
        imgA_d = (img_A[i].cpu() * std + mean).permute(1,2,0).clamp(0,1).numpy()
        imgB_d = (img_B[i].cpu() * std + mean).permute(1,2,0).clamp(0,1).numpy()
        gt     = masks[i, 0].numpy()
        pred   = preds[i, 0].cpu().numpy()

        error  = np.zeros((*pred.shape, 3))
        error[(pred==1)&(gt==1)] = [0,1,0]   # TP green
        error[(pred==1)&(gt==0)] = [1,0,0]   # FP red
        error[(pred==0)&(gt==1)] = [0,0,1]   # FN blue

        for j, (img_d, title) in enumerate(zip([imgA_d, imgB_d, gt, pred, error], titles)):
            if j in [2, 3]:
                axes[i,j].imshow(img_d, cmap='gray')
            else:
                axes[i,j].imshow(img_d)
            axes[i,j].set_title(title)
            axes[i,j].axis('off')

    plt.suptitle("🟢 TP  🔴 FP  🔵 FN", fontsize=12)
    plt.tight_layout()
    plt.savefig("/content/cd_predictions.png", dpi=100)
    plt.show()
    print("Saved cd_predictions.png")

## 🚀 Main — Full Training Pipeline

**Training strategy:**
- Encoder frozen for 5 epochs (warmup)
- Full model fine-tuned from epoch 6
- Early stopping with patience=15
- Checkpoint saved every epoch to Google Drive (resume-safe)
- Best model saved separately by val IoU

**Checkpoint resume:**
If Colab disconnects, simply re-run all cells and `main()` will automatically detect the checkpoint and resume from the last completed epoch.

In [ ]:
def save_checkpoint(model, optimizer, scheduler, epoch, best_iou, no_improve, history, path):
    torch.save({
        'epoch':      epoch,
        'model':      model.state_dict(),
        'optimizer':  optimizer.state_dict(),
        'scheduler':  scheduler.state_dict(),
        'best_iou':   best_iou,
        'no_improve': no_improve,
        'history':    history,
    }, path)

def load_checkpoint(model, optimizer, scheduler, path, device):
    ckpt = torch.load(path, map_location=device, weights_only=False)
    model.load_state_dict(ckpt['model'])
    optimizer.load_state_dict(ckpt['optimizer'])
    scheduler.load_state_dict(ckpt['scheduler'])
    return (
        ckpt['epoch'],
        ckpt['best_iou'],
        ckpt['no_improve'],
        ckpt['history']
    )

def main():
    train_loader, val_loader, test_loader = get_dataloaders()
    print(f"Train: {len(train_loader)} batches | Val: {len(val_loader)} | Test: {len(test_loader)}")

    model     = get_model()
    criterion = CombinedLoss(focal_w=0.35, dice_w=0.40, boundary_w=0.25,
                             alpha=0.75, gamma=2.0).to(CONFIG["device"])

    optimizer = torch.optim.AdamW([
        {"params": model.encoder.parameters(),             "lr": CONFIG["lr"] * 0.1},
        {"params": model.fusion_convs.parameters(),        "lr": CONFIG["lr"]},
        {"params": model.decoder_projections.parameters(), "lr": CONFIG["lr"]},
        {"params": model.decoder_fuse.parameters(),        "lr": CONFIG["lr"]},
        {"params": model.classifier.parameters(),          "lr": CONFIG["lr"]},
    ], weight_decay=0.01)

    scheduler = torch.optim.lr_scheduler.OneCycleLR(
        optimizer,
        max_lr=[CONFIG["lr"] * 0.1, CONFIG["lr"], CONFIG["lr"],
                CONFIG["lr"], CONFIG["lr"]],
        steps_per_epoch=len(train_loader),
        epochs=CONFIG["epochs"],
        pct_start=0.1,
    )

    # ── Resume از checkpoint ──────────────────────────────────────────────
    CHECKPOINT_PATH = "/content/drive/MyDrive/cd_checkpoint.pth"
    BEST_PATH       = "/content/drive/MyDrive/best_cd_model.pth"

    start_epoch = 1
    best_iou    = 0
    no_improve  = 0
    history     = {"train_loss": [], "val_loss": [], "train_iou": [], "val_iou": []}

    if os.path.exists(CHECKPOINT_PATH):
        print(f"▶ Resuming from checkpoint: {CHECKPOINT_PATH}")
        start_epoch, best_iou, no_improve, history = load_checkpoint(
            model, optimizer, scheduler, CHECKPOINT_PATH, CONFIG["device"]
        )
        start_epoch += 1  # epoch بعدی
        print(f"  Resumed from epoch {start_epoch} | Best IoU so far: {best_iou:.4f}")
    else:
        # Freeze encoder for first 5 epochs فقط اگه از اول شروع می‌کنیم
        for param in model.encoder.parameters():
            param.requires_grad = False
        print("Encoder frozen for warmup")

    # ── Training loop ─────────────────────────────────────────────────────
    for epoch in range(start_epoch, CONFIG["epochs"] + 1):

        if epoch == 6 and start_epoch <= 6:
            for param in model.encoder.parameters():
                param.requires_grad = True
            print("✅ Encoder unfrozen")

        # اگه resume از بعد از epoch 5، encoder باید unfreeze باشه
        if start_epoch > 6:
            for param in model.encoder.parameters():
                param.requires_grad = True

        train_loss, train_m = train_epoch(model, train_loader, optimizer,
                                          criterion, scheduler, CONFIG["device"])
        val_loss,   val_m   = eval_epoch(model, val_loader, criterion, CONFIG["device"])

        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        history["train_iou"].append(train_m["iou"])
        history["val_iou"].append(val_m["iou"])

        print(f"Epoch {epoch:02d}/{CONFIG['epochs']} | "
              f"Train Loss: {train_loss:.4f}  IoU: {train_m['iou']:.4f}  Dice: {train_m['dice']:.4f} | "
              f"Val Loss: {val_loss:.4f}  IoU: {val_m['iou']:.4f}  Dice: {val_m['dice']:.4f}")

        if val_m["iou"] > best_iou:
            best_iou   = val_m["iou"]
            no_improve = 0
            torch.save(model.state_dict(), BEST_PATH)
            print(f"  ✓ Saved best model (IoU: {best_iou:.4f})")
        else:
            no_improve += 1
            if no_improve >= CONFIG["patience"]:
                print(f"Early stopping at epoch {epoch}")
                break

        # ← هر epoch checkpoint کامل رو ذخیره می‌کنه
        save_checkpoint(model, optimizer, scheduler, epoch,
                        best_iou, no_improve, history, CHECKPOINT_PATH)

    # ── Test evaluation ───────────────────────────────────────────────────
    print("\n── Test Evaluation ──")
    model.load_state_dict(torch.load(BEST_PATH,
                                     map_location=CONFIG["device"],
                                     weights_only=True))
    test_loss, test_m = eval_epoch(model, test_loader, criterion, CONFIG["device"])
    print(f"Test Loss:      {test_loss:.4f}")
    print(f"Test IoU:       {test_m['iou']:.4f}")
    print(f"Test Dice/F1:   {test_m['dice']:.4f}")
    print(f"Test Precision: {test_m['precision']:.4f}")
    print(f"Test Recall:    {test_m['recall']:.4f}")

    # ── Curves ────────────────────────────────────────────────────────────
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
    ax1.plot(history["train_loss"], label="Train")
    ax1.plot(history["val_loss"],   label="Val")
    ax1.set_title("Loss"); ax1.legend(); ax1.set_xlabel("Epoch")
    ax2.plot(history["train_iou"],  label="Train")
    ax2.plot(history["val_iou"],    label="Val")
    ax2.set_title("IoU");  ax2.legend(); ax2.set_xlabel("Epoch")
    plt.tight_layout()
    plt.savefig("/content/drive/MyDrive/cd_training_curves.png", dpi=100)
    plt.show()

    visualize_predictions(model, test_loader, CONFIG["device"])

if __name__ == "__main__":
    main()

## ▶️ Run Training

In [ ]:
main()

## 🏆 Test Evaluation & Visualization

Loads the best saved model and runs full evaluation on the held-out test set (690 tiles).

**Results:**

| Metric | Score | Std |
|--------|-------|-----|
| IoU | 0.6196 | ±0.1958 |
| Dice / F1 | 0.7458 | ±0.1619 |
| Precision | 0.7322 | — |
| Recall | 0.7696 | — |

The high std on IoU reflects the natural difficulty variation — tiles with dense construction have much higher IoU than sparse rural tiles.

In [ ]:
@torch.no_grad()
def test_model(model, test_loader, criterion, device):
    model.eval()
    total_loss  = 0
    metrics_sum = {"iou": 0, "dice": 0, "precision": 0, "recall": 0}
    all_iou, all_dice = [], []

    pbar = tqdm(test_loader, desc="Testing", dynamic_ncols=True)
    for img_A, img_B, masks in pbar:
        img_A = img_A.to(device)
        img_B = img_B.to(device)
        masks = masks.to(device)

        logits = model(img_A, img_B)
        loss, _ = criterion(logits, masks)

        probs = torch.sigmoid(logits)
        m = compute_metrics(probs, masks)

        total_loss += loss.item()
        for k in metrics_sum:
            metrics_sum[k] += m[k]
        all_iou.append(m["iou"])
        all_dice.append(m["dice"])
        pbar.set_postfix({"IoU": f"{m['iou']:.4f}"})

    n = len(test_loader)
    avg = {k: v / n for k, v in metrics_sum.items()}

    print("\n" + "="*55)
    print("     TEST RESULTS — WHU Building Change Detection")
    print("="*55)
    print(f"  Loss:       {total_loss/n:.4f}")
    print(f"  IoU:        {avg['iou']:.4f}  (±{np.std(all_iou):.4f})")
    print(f"  Dice / F1:  {avg['dice']:.4f}  (±{np.std(all_dice):.4f})")
    print(f"  Precision:  {avg['precision']:.4f}")
    print(f"  Recall:     {avg['recall']:.4f}")
    print("="*55)
    return avg


# ── Run test evaluation ───────────────────────────────────────────────────────
BEST_PATH = "/content/drive/MyDrive/best_cd_model.pth"

model     = get_model()
criterion = CombinedLoss(focal_w=0.35, dice_w=0.40, boundary_w=0.25,
                         alpha=0.75, gamma=2.0).to(CONFIG["device"])

model.load_state_dict(torch.load(BEST_PATH,
                                 map_location=CONFIG["device"],
                                 weights_only=True))
print("Loaded best model from", BEST_PATH)

_, _, test_loader = get_dataloaders()
test_metrics = test_model(model, test_loader, criterion, CONFIG["device"])
visualize_predictions(model, test_loader, CONFIG["device"], n=5)